In [32]:
# Install deps (run once)
import sys
!{sys.executable} -m pip install -q replicate requests pillow



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: /usr/local/Cellar/jupyterlab/4.3.1_1/libexec/bin/python -m pip install --upgrade pip


In [33]:
# Config
from pathlib import Path

# Replicate API token
REPLICATE_API_TOKEN = "YOUR_REPLICATE_API_TOKEN_HERE"

# Paths
BASE_DIR = Path("/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator").resolve()
INPUT_DIR = BASE_DIR / "Photos"
OUTPUT_DIR = BASE_DIR / "photos_wo_background"

INPUT_DIR, OUTPUT_DIR


(PosixPath('/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/Photos'),
 PosixPath('/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/photos_wo_background'))

In [34]:
# Helpers (bria/remove-background)
import os
import re
import base64
from typing import Iterable, Optional, Union

import requests
import replicate
from replicate.helpers import FileOutput

MODEL_REF = "bria/remove-background"  # fixed model slug

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff", ".tif"}


def ensure_output_dir(path: Path) -> None:
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)


def is_supported_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS


def download_to_file(url: str, dest_path: Path) -> None:
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with dest_path.open("wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


def run_bg_removal(image_path: Path) -> Union[str, bytes, Iterable[Union[str, bytes]], dict]:
    with image_path.open("rb") as f:
        # Call by model slug (no version listing)
        return replicate.run(MODEL_REF, input={"image": f})


def _try_decode_base64_image(data: str) -> Optional[bytes]:
    m = re.match(r"^data:image/(png|jpeg|webp);base64,(.+)$", data, re.IGNORECASE)
    b64_part = m.group(2) if m else data
    try:
        decoded = base64.b64decode(b64_part, validate=False)
    except Exception:
        return None
    if decoded.startswith(b"\x89PNG\r\n\x1a\n") or decoded.startswith(b"\xff\xd8\xff") or (decoded[:4] == b"RIFF" and b"WEBP" in decoded[:16]):
        return decoded
    return None


def save_output(output: Union[str, bytes, Iterable[Union[str, bytes]], dict], dest_path: Path) -> None:
    # Unwrap list/tuple
    if isinstance(output, (list, tuple)):
        if not output:
            output = b""
        else:
            # If list of FileOutput, take first
            if isinstance(output[0], FileOutput):
                download_to_file(output[0].url, dest_path)
                return
            output = output[0]
    # Unwrap dict common shapes
    if isinstance(output, dict):
        for key in ("output", "image", "url"):
            if key in output:
                output = output[key]
                break
    # Handle FileOutput
    if isinstance(output, FileOutput):
        download_to_file(output.url, dest_path)
        return
    # Handle string outputs
    if isinstance(output, str):
        if output.startswith("http://") or output.startswith("https://"):
            download_to_file(output, dest_path)
            return
        decoded = _try_decode_base64_image(output)
        if decoded is not None:
            dest_path.write_bytes(decoded)
            return
        raise ValueError("Unexpected string output format; not a URL or base64 image")
    # Handle bytes
    if isinstance(output, (bytes, bytearray)):
        dest_path.write_bytes(bytes(output))
        return
    # Fallback
    raise TypeError(f"Unsupported output type: {type(output)!r}")


In [35]:
print(os.environ["REPLICATE_API_TOKEN"])

YOUR_REPLICATE_API_TOKEN_HERE


In [36]:
# Processing
import os
os.environ["REPLICATE_API_TOKEN"] = REPLICATE_API_TOKEN

ensure_output_dir(OUTPUT_DIR)
images = [p for p in INPUT_DIR.iterdir() if is_supported_image(p)]
print(f"Found {len(images)} images in {INPUT_DIR}")

for idx, image_path in enumerate(images, start=1):
    dest_path = OUTPUT_DIR / f"{image_path.stem}.png"
    print(f"[{idx}/{len(images)}] {image_path.name} -> {dest_path.name}")
    try:
        output = run_bg_removal(image_path)
        save_output(output, dest_path)
    except Exception as exc:
        print(f"  !! Failed: {image_path.name} -> {exc}")

print("Done.")


Found 66 images in /Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/Photos
[1/66] otvertka_minus_012.jpg -> otvertka_minus_012.png
  !! Failed: otvertka_minus_012.jpg -> ReplicateError Details:
status: 401
detail: Invalid token
[2/66] otvertka_plus_003.jpg -> otvertka_plus_003.png
  !! Failed: otvertka_plus_003.jpg -> ReplicateError Details:
status: 401
detail: Invalid token
[3/66] kolovorot_039.jpg -> kolovorot_039.png
  !! Failed: kolovorot_039.jpg -> ReplicateError Details:
status: 401
detail: Invalid token
[4/66] shernitsa_021.jpg -> shernitsa_021.png


KeyboardInterrupt: 

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install -q replicate requests pillow


In [ ]:
# Configuration
from pathlib import Path

# Replicate token
REPLICATE_API_TOKEN = "YOUR_REPLICATE_API_TOKEN_HERE"

# Choose model (owner/model). Stronger alternative by default:
# Try: "briaai/RMBG-1.4". Others: "cjwbw/rembg", "skytnt/carvekit" (if available)
MODEL_NAME = "briaai/RMBG-1.4"
# Optional: pin to a specific version id (or leave empty to use latest)
MODEL_VERSION_ID = ""

# Optional extra inputs for the chosen model (merged into the request)
EXTRA_INPUT = {
    # Example keys may vary by model
    # "background": "transparent",
    # "alpha_matting": True,
}

# Default paths (edit if needed)
BASE_DIR = Path("/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator").resolve()
INPUT_DIR = BASE_DIR / "Photos"
OUTPUT_DIR = BASE_DIR / "photos_wo_background"

INPUT_DIR, OUTPUT_DIR


In [ ]:
# Helpers
import os
import re
import base64
from typing import Iterable, Optional, Union

import requests
import replicate

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff", ".tif"}

# Use explicit Replicate client with provided token to avoid env var issues
client = replicate.Client(api_token=(REPLICATE_API_TOKEN or os.environ.get("REPLICATE_API_TOKEN", "")).strip())


def ensure_output_dir(path: Path) -> None:
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)


def is_supported_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS


def download_to_file(url: str, dest_path: Path) -> None:
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with dest_path.open("wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


_VERSION_REF: Optional[str] = None

def _get_version_ref() -> str:
    global _VERSION_REF
    if _VERSION_REF:
        return _VERSION_REF
    if MODEL_VERSION_ID:
        _VERSION_REF = f"{MODEL_NAME}:{MODEL_VERSION_ID}"
        return _VERSION_REF
    model = client.models.get(MODEL_NAME)
    versions = list(model.versions.list())
    if not versions:
        raise RuntimeError(f"No versions available for {MODEL_NAME} model.")
    _VERSION_REF = f"{MODEL_NAME}:{versions[0].id}"
    return _VERSION_REF


def predict_rembg_output(image_path: Path) -> Union[str, bytes, Iterable[Union[str, bytes]], dict]:
    version_ref = _get_version_ref()
    with image_path.open("rb") as f:
        input_payload = {"image": f}
        if isinstance(EXTRA_INPUT, dict):
            input_payload.update(EXTRA_INPUT)
        return client.run(version_ref, input=input_payload)


def _try_decode_base64_image(data: str) -> Optional[bytes]:
    # Data URI pattern
    m = re.match(r"^data:image/(png|jpeg|webp);base64,(.+)$", data, re.IGNORECASE)
    b64_part = m.group(2) if m else data
    try:
        decoded = base64.b64decode(b64_part, validate=False)
    except Exception:
        return None
    # Check common image magic bytes
    if decoded.startswith(b"\x89PNG\r\n\x1a\n") or decoded.startswith(b"\xff\xd8\xff") or (decoded[:4] == b"RIFF" and b"WEBP" in decoded[:16]):
        return decoded
    return None


def save_output(output: Union[str, bytes, Iterable[Union[str, bytes]], dict], dest_path: Path) -> None:
    # Unwrap list/tuple
    if isinstance(output, (list, tuple)):
        output = output[0] if output else b""
    # Unwrap dict common shapes
    if isinstance(output, dict):
        for key in ("output", "image", "url"):  # best-effort
            if key in output:
                output = output[key]
                break
    # Handle string outputs
    if isinstance(output, str):
        if output.startswith("http://") or output.startswith("https://"):
            download_to_file(output, dest_path)
            return
        decoded = _try_decode_base64_image(output)
        if decoded is not None:
            dest_path.write_bytes(decoded)
            return
        # Unknown string format -> raise to avoid writing junk
        raise ValueError("Unexpected string output format; not a URL or base64 image")
    # Handle bytes
    if isinstance(output, (bytes, bytearray)):
        dest_path.write_bytes(bytes(output))
        return
    # Fallback
    raise TypeError(f"Unsupported output type: {type(output)!r}")


In [ ]:
# Processing via Replicate
import os

os.environ["REPLICATE_API_TOKEN"] = REPLICATE_API_TOKEN

ensure_output_dir(OUTPUT_DIR)
images = [p for p in INPUT_DIR.iterdir() if is_supported_image(p)]
print(f"Model: {MODEL_NAME}")
print(f"Found {len(images)} images in {INPUT_DIR}")

for idx, image_path in enumerate(images, start=1):
    dest_path = OUTPUT_DIR / f"{image_path.stem}.png"
    print(f"[{idx}/{len(images)}] {image_path.name} -> {dest_path.name}")
    try:
        output = predict_rembg_output(image_path)
        save_output(output, dest_path)
    except Exception as exc:
        print(f"  !! Failed: {image_path.name} -> {exc}")

print("Done.")